# Price curves 01: first look at the data

Look before fitting. Plots of wholesale price π against residual load R, and of
the fuel-adjusted ratio π / k against R, for the **training years only**
(2021–2024). The final-test year 2025 stays unseen until `final_test_split`.

Issue #69. No fitting here.

In [ ]:
import os
from collections import Counter
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np

from fleet_sim.prices.config import load_config
from fleet_sim.prices.dataset import build_dataset, selection_data, summarise
from fleet_sim.viz.style import apply_style, style_axes

# The config's cache_dir is relative to the repo root, so run from there.
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
apply_style()

## Load

In [ ]:
config = load_config(Path("calibration/price_curves.toml"))
dataset = selection_data(build_dataset(config), config.splits)

print(f"{'year':>5} {'expected':>9} {'capped':>7} {'no R':>5} {'kept':>6}")
for d in dataset.drops:
    print(
        f"{d.year:>5} {d.expected_slots:>9} {d.capped:>7} "
        f"{d.missing_residual_load:>5} {d.kept:>6}"
    )
print()
print(f"{'year':>5} {'π mean':>7} {'π min':>7} {'π max':>7} {'R min':>6} {'R max':>6} {'k min':>6} {'k max':>6}")
for s in summarise(dataset):
    print(
        f"{s.year:>5} {s.price_mean_gbp_per_mwh:>7.1f} {s.price_min_gbp_per_mwh:>7.1f} "
        f"{s.price_max_gbp_per_mwh:>7.1f} {s.residual_load_min_gw:>6.1f} "
        f"{s.residual_load_max_gw:>6.1f} {s.fuel_cost_min_gbp_per_mwh:>6.1f} "
        f"{s.fuel_cost_max_gbp_per_mwh:>6.1f}"
    )

In [ ]:
obs = dataset.observations
price = np.array([o.wholesale_price_gbp_per_mwh for o in obs])
load = np.array([o.residual_load_gw for o in obs])
cost = np.array([o.fuel_cost_gbp_per_mwh for o in obs])
year = np.array([o.year for o in obs])
start = np.array([o.settlement_start for o in obs])
ratio = price / cost
years = dataset.years
colours = dict(zip(years, plt.get_cmap("viridis")(np.linspace(0, 0.9, len(years)))))

## π against R, by year

Binned medians with the interquartile band, R bins 1 GW wide. Bins with fewer
than 30 points are left out.

In [ ]:
BIN_GW = 1.0
MIN_POINTS = 30
edges = np.arange(np.floor(load.min()), np.ceil(load.max()) + BIN_GW, BIN_GW)
centres = (edges[:-1] + edges[1:]) / 2


def binned(x, y):
    """Median and quartiles of y in each R bin; NaN where a bin is thin."""
    index = np.digitize(x, edges) - 1
    out = np.full((3, len(centres)), np.nan)
    for b in range(len(centres)):
        inside = y[index == b]
        if len(inside) >= MIN_POINTS:
            out[:, b] = np.percentile(inside, [25, 50, 75])
    return out


def plot_by_year(y, ylabel):
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    for yr in years:
        mask = year == yr
        q25, q50, q75 = binned(load[mask], y[mask])
        ax.fill_between(centres, q25, q75, color=colours[yr], alpha=0.18, linewidth=0)
        ax.plot(centres, q50, color=colours[yr], label=str(yr))
    ax.set_xlabel("Residual load R (GW)")
    ax.set_ylabel(ylabel)
    ax.legend()
    style_axes(ax)
    return fig, ax


plot_by_year(price, "Wholesale price π (£/MWh)");

In [ ]:
def hexbin_by_year(y, ylabel):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
    for ax, yr in zip(axes.flat, years):
        mask = year == yr
        ax.hexbin(load[mask], y[mask], gridsize=50, bins="log", cmap="viridis", mincnt=1)
        ax.set_title(str(yr))
        style_axes(ax)
    for ax in axes[1]:
        ax.set_xlabel("R (GW)")
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel)
    fig.tight_layout()


hexbin_by_year(price, "π (£/MWh)")

## π / k against R, by year

In [ ]:
plot_by_year(ratio, "π / k");

In [ ]:
hexbin_by_year(ratio, "π / k")

## π, k and R over time

Grey is half-hourly; the line is the daily mean.

In [ ]:
day = np.array([s.date() for s in start])
days = sorted(set(day))
day_index = {d: i for i, d in enumerate(days)}
which_day = np.array([day_index[d] for d in day])
counts = np.bincount(which_day)

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for ax, y, label in zip(
    axes,
    [price, cost, load],
    ["π (£/MWh)", "k (£/MWh)", "R (GW)"],
):
    daily = np.bincount(which_day, weights=y) / counts
    ax.plot(start, y, color="#9aa5b1", linewidth=0.3)
    ax.plot(days, daily, color="#1f2933", linewidth=1.0)
    ax.set_ylabel(label)
    style_axes(ax)
fig.tight_layout()

## π / k by hour of day

Hours are UK local time. Median line, interquartile band.

In [ ]:
london = ZoneInfo("Europe/London")
hour = np.array([s.astimezone(london).hour for s in start])

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for yr in years:
    mask = year == yr
    q = np.array([np.percentile(ratio[mask & (hour == h)], [25, 50, 75]) for h in range(24)]).T
    ax.fill_between(range(24), q[0], q[2], color=colours[yr], alpha=0.18, linewidth=0)
    ax.plot(range(24), q[1], color=colours[yr], label=str(yr))
ax.set_xlabel("Hour of day (UK local)")
ax.set_ylabel("π / k")
ax.set_xticks(range(0, 24, 3))
ax.legend()
style_axes(ax)

## Data checks

In [ ]:
print("negative π:", Counter(year[price < 0].tolist()))
print("zero π:", Counter(year[price == 0].tolist()))
print("negative R:", Counter(year[load < 0].tolist()))
print("π / k percentiles [0, 0.1, 1, 50, 99, 99.9, 100]:")
print(np.round(np.percentile(ratio, [0, 0.1, 1, 50, 99, 99.9, 100]), 2))

seconds = np.array([s.timestamp() for s in start])
gap_h = np.diff(seconds) / 3600
print()
print(f"gaps longer than one slot: {(gap_h > 0.5).sum()}")
for i in np.argsort(gap_h)[::-1][:8]:
    print(f"  {start[i]:%Y-%m-%d %H:%M} UTC  {gap_h[i]:.1f} h")

In [ ]:
# Where capping falls: the highest kept price each year sits just under the cap,
# and the capped slots are missing from the top of the curve.
for d in dataset.drops:
    mask = year == d.year
    print(
        f"{d.year}: capped {d.capped / d.expected_slots:.0%} of slots, "
        f"π max kept {price[mask].max():.0f}, "
        f"slots with R above 30 GW kept: {(load[mask] > 30).sum()}"
    )

In [ ]:
# Exact prices that repeat hundreds of times are a sign of a retail ceiling.
# A peak slot at value v and an off-peak slot at v + 60 £/MWh have the same
# Agile retail rate in region C (multiplier 2, peak adder 12 p/kWh).
for yr in years:
    mask = year == yr
    values, n = np.unique(np.round(price[mask], 2), return_counts=True)
    top = np.argsort(n)[::-1][:4]
    print(yr, [(float(values[i]), int(n[i])) for i in top])

for value in [110.46, 170.46, 226.41, 286.41]:
    at = np.isclose(price, value, atol=0.01)
    months = Counter(s.strftime("%Y-%m") for s in start[at])
    print(f"{value}: {at.sum()} slots, months {sorted(months)}")

## Summary

Training years 2021–2024 only; 2025 is not looked at.

**Do the yearly bands collapse under π / k?** Yes, over most of the range. In
π against R the yearly medians sit in separate bands: at 20 GW, 2022 is near
£160/MWh and 2024 near £85/MWh. In π / k against R the four medians lie within
about 0.1 of each other from 5 to 28 GW. The fuel adjustment looks worth
carrying into the fit. Two places don't collapse:

- 2022 above 30 GW, where π / k falls back to about 1. These are December 2022
  cold-snap slots at a retail ceiling the dataset doesn't flag (see below), so
  this is a data problem, not a shape.
- 2021 runs a little high from 22 to 28 GW (about 1.3 against 1.2). 2021 is a
  part year with few slots above 25 GW, and a third of its slots are capped.

**Plausible shapes for π / k against R.** Three regimes:

1. Below about 9 GW, a steep rise from about 0 (and sometimes negative) to 0.85.
2. From 9 to about 28 GW, close to linear and shallow, rising about 0.015 per
   GW from 0.85 to 1.2.
3. Above about 30 GW, steeper again, reaching 1.5–1.6 by 37 GW in 2023–2024.
   Data here is thin.

That is concave at low R and convex at high R. Candidates for #70: a cubic in
R, piecewise linear with knots near 9 and 30 GW, and a linear-plus-exponential
tail. A quadratic can't bend both ways. Any family must allow π / k ≤ 0 at
low R: negative prices are real and are growing, from 2 slots in 2021 to 322
in 2024.

π / k by hour of day has morning and evening peaks, but R peaks at the same
hours, so this is not evidence of an hour effect beyond R. Look at fit
residuals by hour after fitting.

**Anything odd in the data.**

- **Undetected retail ceiling, October 2022 to June 2023 (data problem; new
  issue).** In region C, AGILE-22-08-31 rates repeat at 34.09 p/kWh excluding
  VAT, and at 57.28 p/kWh in November–December 2022. These are hard ceilings
  in the raw Agile data, and a peak slot and an off-peak slot invert to the
  same retail rate (π = 110.46 and 170.46, and 226.41 and 286.41 £/MWh). The
  code assumes a 78p cap including VAT for that product, so these slots are
  not flagged as capped. About 2,100 censored prices are in the training data,
  and 2023 reports 0 capped slots. They pull the top of the 2022 and 2023
  curves down.
- **Capping is not random (modelling note for #71).** About a third of 2021
  and 2022 slots are dropped as capped, and these are the highest-price slots.
  The longest gaps, of 4–12 days in December 2021 and February–March 2022,
  are whole capped stretches during the gas crisis. Fitting only the kept
  slots biases those years low at high R and high k. A censored fit, or
  down-weighting those years at the top, may be needed.
- Negative R in 4 slots in 2023 (down to −0.5 GW). Plausible: wind plus
  embedded generation briefly exceeding demand. Not an error.
- No sign errors seen: k is always positive, and negative π falls at low R.